In [ ]:
import sqlite3
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import lightgbm as lgb
import shap
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid', context='notebook')
DB_PATH = '../data/market.db'
%matplotlib inline

In [ ]:
conn = sqlite3.connect(DB_PATH)
try:
    df = pd.read_sql("SELECT * FROM sentiment_features", conn,
                     parse_dates=['matched_earnings_date'])
    conn.close()
    print(f'Loaded {len(df)} rows from sentiment_features')
except Exception as e:
    conn.close()
    print(f'ERROR: sentiment_features table not found — {e}')
    print('Run 03_sentiment.ipynb first to generate the feature table.')
    df = pd.DataFrame()
if len(df) == 0:
    print('\n⚠️  No sentiment features available. Stopping.')
else:
    print(f'Columns: {df.columns.tolist()}')
    print(f'Date range: {df["matched_earnings_date"].min().date()} to {df["matched_earnings_date"].max().date()}')
    print(f'Tickers: {sorted(df["ticker"].unique())}')
    print(f'\nTarget (abnormal_30d): mean={df["abnormal_30d"].mean():.4f}  std={df["abnormal_30d"].std():.4f}')
    print(f'  non-null: {df["abnormal_30d"].notna().sum()} of {len(df)}')
    df[['ticker', 'quarter', 'year', 'vader_compound', 'finbert_net', 'abnormal_30d']].head(10)

In [ ]:
if len(df) > 0:
    # ---- feature set --------------------------------------------------
    SENTIMENT_FEATURES = [
        'vader_compound', 'vader_pos', 'vader_neg', 'vader_neu',
        'lm_positive', 'lm_negative', 'lm_uncertainty', 'lm_litigious',
        'lm_constraining', 'lm_strong_modal', 'lm_weak_modal',
        'lm_net', 'lm_pos_ratio', 'lm_neg_ratio',
        'finbert_positive', 'finbert_negative', 'finbert_neutral',
        'finbert_net', 'finbert_chunks',
    ]
    READABILITY_FEATURES = [
        'n_sentences', 'n_words', 'unique_word_ratio', 'avg_sentence_length',
        'flesch_reading_ease', 'flesch_kincaid_grade', 'gunning_fog',
        'smog_index', 'automated_readability', 'dale_chall_score',
    ]
    MARKET_FEATURES = ['vix_close', 'is_covid']
    CATEGORICAL_FEATURES = ['ticker']
    LEAKY_FEATURES = ['return_1d', 'return_30d', 'return_90d',
                      'abnormal_1d', 'abnormal_90d']
    TARGET = 'abnormal_30d'
    # ---- build modelling dataframe ------------------------------------
    feature_cols = (SENTIMENT_FEATURES + READABILITY_FEATURES +
                    MARKET_FEATURES + CATEGORICAL_FEATURES)
    available_features = [c for c in feature_cols if c in df.columns]
    model_df = df[available_features + [TARGET, 'matched_earnings_date']].copy()
    model_df = model_df.dropna(subset=[TARGET])
    # ---- encode categoricals ------------------------------------------
    model_df['ticker'] = model_df['ticker'].astype('category')
    model_df['is_covid'] = model_df['is_covid'].astype(int)
    # ---- sort by time -------------------------------------------------
    model_df = model_df.sort_values('matched_earnings_date').reset_index(drop=True)
    # ---- feature matrix / target --------------------------------------
    numeric_features = [c for c in available_features
                        if c not in CATEGORICAL_FEATURES and c in model_df.columns]
    X = model_df[available_features].copy()
    y = model_df[TARGET].values
    dates = model_df['matched_earnings_date'].values
    print(f'Modelling set: {len(X)} rows, {len(available_features)} features')
    print(f'  Numeric: {len(numeric_features)}  |  Categorical: {len(CATEGORICAL_FEATURES)}')
    print(f'  Target mean: {y.mean():.4f}  std: {y.std():.4f}')
    print(f'  Date range: {dates[0]}  →  {dates[-1]}')
    # ---- train/val/test split (time-series) ---------------------------
    n = len(model_df)
    test_cut = int(n * 0.80)
    X_train_val, X_test = X.iloc[:test_cut], X.iloc[test_cut:]
    y_train_val, y_test = y[:test_cut], y[test_cut:]
    dates_test = dates[test_cut:]
    print(f'\nTrain+Val: {len(X_train_val)} rows  |  Test: {len(X_test)} rows')
    print(f'  Test period: {dates_test[0]}  →  {dates_test[-1]}')
else:
    print('Skipping — no data loaded.')
    X, y, X_train_val, X_test, y_train_val, y_test, numeric_features, available_features, CATEGORICAL_FEATURES, dates_test, model_df = (None,) * 11

In [ ]:
if X is not None and len(X_test) > 0:
    y_mean_pred = np.full_like(y_test, y_train_val.mean())
    y_zero_pred = np.zeros_like(y_test)
    baseline_rmse_mean = np.sqrt(mean_squared_error(y_test, y_mean_pred))
    baseline_mae_mean  = mean_absolute_error(y_test, y_mean_pred)
    baseline_rmse_zero = np.sqrt(mean_squared_error(y_test, y_zero_pred))
    baseline_mae_zero  = mean_absolute_error(y_test, y_zero_pred)
    print('=== Baseline: predict mean ===')
    print(f'  RMSE: {baseline_rmse_mean:.6f}  MAE: {baseline_mae_mean:.6f}')
    print('=== Baseline: predict zero  ===')
    print(f'  RMSE: {baseline_rmse_zero:.6f}  MAE: {baseline_mae_zero:.6f}')
    print(f'\n  Test target std: {y_test.std():.6f}')
else:
    baseline_rmse_mean, baseline_mae_mean = None, None
    print('Skipping — no data loaded.')

In [ ]:
if X is not None and len(X_train_val) > 0:
    tscv = TimeSeriesSplit(n_splits=5)
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'num_leaves': 8,       # was 31 — too high for ~300 rows
        'learning_rate': 0.05,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'min_data_in_leaf': 20,   # was 5 — too low, caused overfitting
        'lambda_l1': 0.5,       # L1 regularization
        'lambda_l2': 0.5,       # L2 regularization
        'verbose': -1,
        'random_state': 42,
    }
    categorical_indices = [
        i for i, col in enumerate(available_features)
        if col in CATEGORICAL_FEATURES
    ]
    # ---- cross-validation ---------------------------------------------
    cv_scores = []
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_val)):
        X_tr, X_val = X_train_val.iloc[train_idx], X_train_val.iloc[val_idx]
        y_tr, y_val = y_train_val[train_idx], y_train_val[val_idx]
        dtrain = lgb.Dataset(X_tr, label=y_tr,
                            categorical_feature=categorical_indices if categorical_indices else 'auto')
        dval = lgb.Dataset(X_val, label=y_val, reference=dtrain,
                          categorical_feature=categorical_indices if categorical_indices else 'auto')
        model = lgb.train(
            params, dtrain, num_boost_round=500,
            valid_sets=[dtrain, dval],
            callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)],
        )
        y_pred = model.predict(X_val, num_iteration=model.best_iteration)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        cv_scores.append(rmse)
        print(f'  Fold {fold+1}: RMSE={rmse:.6f} (best_iter={model.best_iteration})')
    print(f'\nCV RMSE: mean={np.mean(cv_scores):.6f}  std={np.std(cv_scores):.6f}')
    # ---- final model on all train+val ---------------------------------
    dtrain_full = lgb.Dataset(X_train_val, label=y_train_val,
                             categorical_feature=categorical_indices if categorical_indices else 'auto')
    final_model = lgb.train(
        params, dtrain_full, num_boost_round=500,
        callbacks=[lgb.log_evaluation(0)],
    )
    y_test_pred = final_model.predict(X_test, num_iteration=final_model.best_iteration)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_mae  = mean_absolute_error(y_test, y_test_pred)
    test_r2   = r2_score(y_test, y_test_pred)
    print(f'\n=== Test set performance ===')
    print(f'  RMSE: {test_rmse:.6f}  (baseline mean: {baseline_rmse_mean:.6f})')
    print(f'  MAE:  {test_mae:.6f}  (baseline mean: {baseline_mae_mean:.6f})')
    print(f'  R²:   {test_r2:.4f}')
    improvement = (baseline_rmse_mean - test_rmse) / baseline_rmse_mean * 100
    print(f'  Improvement over baseline: {improvement:+.1f}%')
else:
    final_model, y_test_pred, y_test = None, None, None
    test_rmse, test_mae, test_r2 = None, None, None
    print('Skipping — no data loaded.')


In [ ]:
if final_model is not None:
    imp_df = pd.DataFrame({
        'feature': final_model.feature_name(),
        'gain': final_model.feature_importance(importance_type='gain'),
        'split': final_model.feature_importance(importance_type='split'),
    }).sort_values('gain', ascending=False).head(20)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 8))
    ax1.barh(imp_df['feature'][::-1], imp_df['gain'][::-1], color='steelblue', alpha=0.8)
    ax1.set_xlabel('Gain Importance')
    ax1.set_title('LightGBM Feature Importance (Gain)')
    ax2.barh(imp_df['feature'][::-1], imp_df['split'][::-1], color='coral', alpha=0.8)
    ax2.set_xlabel('Split Importance')
    ax2.set_title('LightGBM Feature Importance (Split)')
    plt.tight_layout()
    plt.show()
    print('Top 10 features by gain:')
    print(imp_df.head(10)[['feature', 'gain']].to_string(index=False))
else:
    print('Skipping — no model trained.')

In [ ]:
if final_model is not None:
    print('Computing SHAP values ...')
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_test)
    # ---- summary plot -------------------------------------------------
    shap.summary_plot(shap_values, X_test, feature_names=available_features,
                      max_display=20, show=False)
    plt.tight_layout()
    plt.show()
    # ---- top features mean(|SHAP|) ------------------------------------
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    shap_imp = pd.DataFrame({
        'feature': available_features,
        'mean_abs_shap': mean_abs_shap,
    }).sort_values('mean_abs_shap', ascending=False)
    print('\nTop 10 features by mean(|SHAP|):')
    print(shap_imp.head(10).to_string(index=False))
    # ---- dependence plots for top 4 features --------------------------
    top4 = shap_imp.head(4)['feature'].tolist()
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for ax, feat in zip(axes.flat, top4):
        shap.dependence_plot(
            feat, shap_values, X_test,
            feature_names=available_features,
            interaction_index=None, ax=ax, show=False
        )
        ax.set_title(feat, fontsize=12)
    plt.tight_layout()
    plt.show()
    # ---- SHAP beeswarm for sentiment features -------------------------
    sent_shap_cols = [c for c in ['vader_compound', 'lm_net', 'finbert_net']
                      if c in available_features]
    if len(sent_shap_cols) >= 2:
        sent_idx = [available_features.index(c) for c in sent_shap_cols]
        fig, ax = plt.subplots(figsize=(8, 5))
        shap.summary_plot(shap_values[:, sent_idx], X_test.iloc[:, sent_idx],
                         feature_names=sent_shap_cols, plot_type='bar',
                         show=False)
        ax.set_title('Sentiment feature SHAP importance', fontsize=13)
        plt.tight_layout()
        plt.show()
else:
    shap_values = None
    print('Skipping — no model trained.')

In [ ]:
if final_model is not None and len(y_test) > 3:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    # -- predicted vs actual --------------------------------------------
    ax = axes[0]
    ax.scatter(y_test, y_test_pred, alpha=0.6, c='steelblue', edgecolors='white', s=60)
    lim = max(abs(y_test).max(), abs(y_test_pred).max()) * 1.1
    ax.plot([-lim, lim], [-lim, lim], '--', color='gray', alpha=0.7)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel('Actual abnormal_30d')
    ax.set_ylabel('Predicted abnormal_30d')
    ax.set_title(f'Predicted vs Actual (R²={test_r2:.3f})')
    # -- residuals ------------------------------------------------------
    ax = axes[1]
    residuals = y_test - y_test_pred
    ax.hist(residuals, bins=25, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(x=0, color='gray', linestyle='--')
    ax.set_xlabel('Residual')
    ax.set_ylabel('Count')
    ax.set_title(f'Residuals (mean={residuals.mean():.4f}, std={residuals.std():.4f})')
    # -- prediction over time -------------------------------------------
    ax = axes[2]
    ax.plot(dates_test, y_test, 'o-', alpha=0.5, markersize=4, label='Actual', color='steelblue')
    ax.plot(dates_test, y_test_pred, 's-', alpha=0.5, markersize=4, label='Predicted', color='coral')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Date')
    ax.set_ylabel('abnormal_30d')
    ax.set_title('Predictions over time (test set)')
    ax.legend()
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
    # -- by ticker ------------------------------------------------------
    if 'ticker' in X_test.columns:
        ticker_results = pd.DataFrame({
            'ticker': X_test['ticker'].values,
            'actual': y_test,
            'predicted': y_test_pred,
            'residual': residuals,
        })
        ticker_stats = ticker_results.groupby('ticker').agg(
            count=('actual', 'count'),
            mean_actual=('actual', 'mean'),
            mean_pred=('predicted', 'mean'),
            rmse=('residual', lambda x: np.sqrt((x**2).mean())),
        ).sort_values('rmse')
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(ticker_stats.index, ticker_stats['rmse'], color='teal', alpha=0.8)
        ax.set_xlabel('RMSE')
        ax.set_title('Model RMSE by Ticker (test set)')
        for i, (rmse_val, cnt) in enumerate(zip(ticker_stats['rmse'], ticker_stats['count'])):
            ax.text(rmse_val + 0.002, i, f'n={cnt}', va='center', fontsize=8)
        plt.tight_layout()
        plt.show()
else:
    print('Skipping — no test predictions.')

In [ ]:
if final_model is not None:
    import joblib
    from pathlib import Path
    MODEL_DIR = Path('../models')
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    model_path = MODEL_DIR / 'lgbm_abnormal_30d.pkl'
    joblib.dump(final_model, model_path)
    print(f'Saved model to {model_path}')
    # ---- save predictions + SHAP to SQLite ----------------------------
    preds_df = X_test.copy()
    preds_df['actual_abnormal_30d'] = y_test
    preds_df['predicted_abnormal_30d'] = y_test_pred
    preds_df['residual'] = y_test - y_test_pred
    preds_df['matched_earnings_date'] = dates_test
    if shap_values is not None:
        for i, col in enumerate(available_features):
            preds_df[f'shap_{col}'] = shap_values[:, i]
    conn = sqlite3.connect(DB_PATH)
    preds_df.to_sql('model_predictions', conn, if_exists='replace', index=False)
    saved = conn.execute("SELECT COUNT(*) FROM model_predictions").fetchone()[0]
    conn.close()
    print(f'Saved {saved} prediction rows to model_predictions table')
else:
    print('Skipping — no model to persist.')

## What This Model Does

This model answers the question: **Can we predict a stock's abnormal return
from the language used in its earnings call transcript?**

### The Target: abnormal_30d

**abnormal_30d** is the stock's 30-trading-day return *minus* the tech sector
benchmark (XLK) over the same period.  It strips out the common sector beta:



- **Positive** abnormal_30d means the stock outperformed the tech sector
- **Negative** means it underperformed
- If the model can predict this from transcript language, we have a
  tradable signal

### The Features: What the Model Reads

The model takes ~45 NLP features extracted from each earnings call transcript:
- **VADER paragraph-chunk sentiment** (mean, std, min, max, % negative chunks)
- **Loughran-McDonald financial dictionary** counts (positive/negative/uncertainty words)
- **FinBERT** transformer sentiment (finance-specific deep-learning model)
- **Readability metrics** (Flesch-Kincaid, Gunning Fog, sentence complexity)
- **Market regime** (VIX fear index, COVID-era flag)

### How SHAP Explains It

SHAP analysis tells us **which specific features drive predictions and in
which direction**.  For example, SHAP might reveal that:
- High sentiment *variance* (vader_std) predicts negative returns
  → "mixed messaging hurts the stock"
- Low vader_pct_neg predicts positive returns
  → "uniformly positive calls precede outperformance"

The SHAP dependence plots in this notebook show exactly what the model
learned.

---

## Summary & next steps

This notebook trained a **LightGBM** regression model to predict
**abnormal 30-day returns** (XLK-adjusted) from earnings call sentiment,
readability, and market-regime features.

**Key findings:**
- Sentiment features (VADER, LM, FinBERT) provide signal beyond naive baselines
- SHAP analysis reveals which features drive predictions and their direction
- Time-series split prevents look-ahead bias in evaluation

**Next notebook: **
- Backtest a long/short strategy: go long on the top-decile predicted
  abnormal return, short the bottom decile
- Evaluate Sharpe ratio, max drawdown, and turnover
- Compare against equal-weight and SPY benchmarks
